# Hybrid MPI–CUDA & Cache-Line Coherency
Executed hardware evidence and analytical responses. GitHub: 240103149-lgtm

In [1]:
# LAB_CELL_1
import os, subprocess, sys
RUN_LOGS = {}
def command(argv):
    p = subprocess.run(argv, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(p.stdout)
    RUN_LOGS[1] = RUN_LOGS.get(1, "") + p.stdout
    if p.returncode:
        raise RuntimeError(f"Command failed: {argv}")
command(["nvidia-smi", "--query-gpu=name,uuid,compute_cap,memory.total", "--format=csv"])
command(["lscpu"])
if os.path.exists("/content"):
    command(["apt-get", "-qq", "update"])
    command(["apt-get", "-qq", "install", "-y", "openmpi-bin", "libopenmpi-dev"])
command([sys.executable, "-m", "pip", "install", "-q", "mpi4py", "numba-cuda[cu12]", "numpy", "nbformat"])

name, uuid, compute_cap, memory.total [MiB]
Tesla T4, GPU-639848cb-ba85-af70-c282-e9deed9ac09b, 7.5, 15360 MiB
Architecture:                            x86_64
CPU op-mode(s):                          32-bit, 64-bit
Address sizes:                           46 bits physical, 48 bits virtual
Byte Order:                              Little Endian
CPU(s):                                  2
On-line CPU(s) list:                     0,1
Vendor ID:                               GenuineIntel
Model name:                              Intel(R) Xeon(R) CPU @ 2.00GHz
CPU family:                              6
Model:                                   85
Thread(s) per core:                      2
Core(s) per socket:                      1
Socket(s):                               1
Stepping:                                3
BogoMIPS:                                4000.27
Flags:                                   fpu vme de pse tsc msr pae mce cx8 apic sep mtrr pge mca cmov pat pse36 clflush mmx fxsr sse

In [2]:
# LAB_CELL_2
GITHUB_USERNAME = "240103149-lgtm"
assert GITHUB_USERNAME.strip() and GITHUB_USERNAME != "REPLACE_WITH_YOUR_GITHUB_USERNAME"
RUN_LOGS[2] = f"Verified solo student handle: {GITHUB_USERNAME}\n"
print(RUN_LOGS[2])

Verified solo student handle: 240103149-lgtm


In [3]:
%%writefile hybrid_mpi_cuda_cache.py
import argparse
import hashlib
import json
import math
import os
from pathlib import Path
import platform
import re
import subprocess
import sys
import time

import numpy as np
from mpi4py import MPI
from numba import cuda, float64, njit, types
from numba.extending import intrinsic
from llvmlite import ir

TILE_DIM = 16
SHARED_DIM = 18
GLOBAL_NX = GLOBAL_NY = 512
LOCAL_N = 256
STENCIL_STEPS = 300
SHM_ITERS = 40_000_000
LAMBDA_DAMP = 0.15
DX = 1.0 / 513.0


def gpu_info():
    raw = subprocess.check_output([
        'nvidia-smi', '--query-gpu=name,uuid,compute_cap',
        '--format=csv,noheader'], text=True).strip().splitlines()[0]
    name, uuid, capability = [v.strip() for v in raw.split(',')]
    return {'name': name, 'uuid': uuid, 'compute_capability': capability}


def cpu_info():
    info = subprocess.check_output(['lscpu', '-J'], text=True)
    topology = json.loads(info)
    stable = [row for row in topology['lscpu'] if row['field'].rstrip(':') in
              ('Architecture', 'CPU(s)', 'On-line CPU(s) list', 'Vendor ID', 'Model name',
               'Model', 'CPU family', 'Stepping', 'Thread(s) per core', 'Core(s) per socket',
               'Socket(s)', 'L1d cache', 'L1i cache', 'L2 cache', 'L3 cache', 'NUMA node(s)')]
    signature = hashlib.sha256(json.dumps(stable, sort_keys=True).encode()).hexdigest()
    line_path = Path('/sys/devices/system/cpu/cpu0/cache/index0/coherency_line_size')
    line = int(line_path.read_text()) if line_path.exists() else 64
    return {'signature': signature, 'topology': topology, 'cache_line_bytes': line,
            'logical_cpus': os.cpu_count(), 'architecture': platform.machine()}


def shared_context(comm):
    node = comm.Split_type(MPI.COMM_TYPE_SHARED)
    if node.size != 4:
        raise RuntimeError('All four ranks must share one compute node.')
    win = MPI.Win.Allocate_shared(64 * 8 + 63 if node.rank == 0 else 0, 1, comm=node)
    buf, _ = win.Shared_query(0)
    raw = np.ndarray(buffer=buf, dtype=np.uint8, shape=(64 * 8 + 63,))
    offset = node.bcast((-raw.ctypes.data) % 64 if node.rank == 0 else None, root=0)
    view = np.ndarray(buffer=buf, dtype=np.uint64, offset=offset, shape=(64,))
    if view.ctypes.data % 64:
        raise RuntimeError('Shared window is not locally aligned to 64 bytes.')
    win.Lock_all()
    if node.rank == 0:
        view[:] = 0
    win.Sync()
    node.Barrier()
    win.Sync()
    return node, win, view


def init_hybrid_context(github_user):
    comm = MPI.COMM_WORLD
    if comm.size != 4:
        raise RuntimeError(f'Requires exactly 4 MPI ranks; got {comm.size}.')
    if os.environ.get('NUMBA_ENABLE_CUDASIM') == '1':
        raise RuntimeError('Simulator is prohibited for official telemetry.')
    cuda.select_device(0)
    gpu = comm.bcast(gpu_info() if comm.rank == 0 else None, root=0)
    own_uuid = gpu_info()['uuid']
    if not comm.allreduce(own_uuid == gpu['uuid'], op=MPI.LAND):
        raise RuntimeError('Every rank must use the same physical GPU 0.')
    cpu = comm.bcast(cpu_info() if comm.rank == 0 else None, root=0)
    if cpu['cache_line_bytes'] != 64:
        raise RuntimeError('This experiment requires 64-byte CPU cache lines.')
    cart = comm.Create_cart((2, 2), periods=(False, False), reorder=False)
    up, down = cart.Shift(0, 1)
    left, right = cart.Shift(1, 1)
    node, win, view = shared_context(comm)
    material = f'{github_user.strip()}::{cpu["signature"]}::{gpu["uuid"]}'
    seed = int.from_bytes(hashlib.sha256(material.encode()).digest()[:8], 'big')
    return dict(comm=comm, cart_comm=cart, rank=comm.rank, coords=cart.Get_coords(comm.rank),
                neighbors=(up, down, left, right), node_comm=node, win=win,
                shm_view=view, cpu=cpu, gpu=gpu, INSTANCE_SEED=seed,
                github_username=github_user.strip())


def initialize_fields(ctx):
    y0, x0 = ctx['coords']
    iy = y0 * LOCAL_N + np.arange(258)
    ix = x0 * LOCAL_N + np.arange(258)
    y, x = np.meshgrid(iy * DX, ix * DX, indexing='ij')
    phase = (ctx['INSTANCE_SEED'] % 1_000_003) / 1_000_003 * 2 * np.pi
    u = np.sin(np.pi * x) * np.sin(np.pi * y)
    u *= 1 + 0.05 * np.cos(2 * np.pi * x + phase) * np.cos(2 * np.pi * y)
    physical = (iy[:, None] == 0) | (iy[:, None] == 513) | (ix[None, :] == 0) | (ix[None, :] == 513)
    u[physical] = 0
    kappa = 0.1 + 0.025 * np.sin(2 * np.pi * x + phase) * np.sin(2 * np.pi * y)
    source = 0.01 * (1 + 0.2 * np.cos(2 * np.pi * x + phase) * np.cos(2 * np.pi * y))
    return tuple(np.ascontiguousarray(v, dtype=np.float64) for v in (u, kappa, source))


@cuda.jit
def pack_halos_kernel(u, packed):
    t = cuda.grid(1)
    if t < LOCAL_N:
        j = t + 1
        packed[0, t] = u[1, j]
        packed[1, t] = u[LOCAL_N, j]
        packed[2, t] = u[j, 1]
        packed[3, t] = u[j, LOCAL_N]


@cuda.jit
def unpack_halos_kernel(u, packed, up, down, left, right):
    t = cuda.grid(1)
    if t < LOCAL_N:
        j = t + 1
        if up:
            u[0, j] = packed[0, t]
        if down:
            u[257, j] = packed[1, t]
        if left:
            u[j, 0] = packed[2, t]
        if right:
            u[j, 257] = packed[3, t]


@cuda.jit
def tiled_stencil_kernel(u, out, kappa, source, dt, inv_dx2, region):
    tile = cuda.shared.array((SHARED_DIM, SHARED_DIM), float64)
    tx, ty = cuda.threadIdx.x, cuda.threadIdx.y
    bx, by = cuda.blockIdx.x * TILE_DIM, cuda.blockIdx.y * TILE_DIM
    tid = ty * TILE_DIM + tx
    for q in range(tid, SHARED_DIM * SHARED_DIM, TILE_DIM * TILE_DIM):
        sy, sx = q // SHARED_DIM, q % SHARED_DIM
        yi, xi = by + sy, bx + sx
        if region == 1 and (yi == 0 or yi == 257 or xi == 0 or xi == 257):
            tile[sy, sx] = 0.0
        else:
            tile[sy, sx] = u[yi, xi]
    cuda.syncthreads()
    i, j = by + ty + 1, bx + tx + 1
    border = i == 1 or i == LOCAL_N or j == 1 or j == LOCAL_N
    if region == 0 or (region == 1 and not border) or (region == 2 and border):
        sy, sx = ty + 1, tx + 1
        v = tile[sy, sx]
        k = kappa[i, j]
        flux = (0.5 * (k + kappa[i-1, j]) * (tile[sy-1, sx] - v)
                + 0.5 * (k + kappa[i+1, j]) * (tile[sy+1, sx] - v)
                + 0.5 * (k + kappa[i, j-1]) * (tile[sy, sx-1] - v)
                + 0.5 * (k + kappa[i, j+1]) * (tile[sy, sx+1] - v)) * inv_dx2
        out[i, j] = v + dt * (flux + source[i, j] - LAMBDA_DAMP * v)


class Pipeline:
    def __init__(self, ctx, host_u, host_k, host_s, dt):
        self.ctx, self.dt = ctx, dt
        self.interior, self.boundary = cuda.stream(), cuda.stream()
        self.u = cuda.to_device(host_u)
        self.out = cuda.to_device(host_u)
        self.k = cuda.to_device(host_k)
        self.source = cuda.to_device(host_s)
        self.send_d = cuda.device_array((4, LOCAL_N), np.float64)
        self.recv_d = cuda.device_array((4, LOCAL_N), np.float64)
        self.send_h = cuda.pinned_array((4, LOCAL_N), np.float64)
        self.recv_h = cuda.pinned_array((4, LOCAL_N), np.float64)
        self.recv_h[:] = 0
        self.present = tuple(int(n != MPI.PROC_NULL) for n in ctx['neighbors'])

    def exchange_start(self):
        c = self.ctx['cart_comm']
        requests = []
        opposite = (1, 0, 3, 2)
        for d, neighbor in enumerate(self.ctx['neighbors']):
            if neighbor != MPI.PROC_NULL:
                requests.append(c.Irecv(self.recv_h[d], source=neighbor, tag=100+opposite[d]))
        pack_halos_kernel[2, 128, self.boundary](self.u, self.send_d)
        self.send_d.copy_to_host(self.send_h, stream=self.boundary)
        self.boundary.synchronize()
        for d, neighbor in enumerate(self.ctx['neighbors']):
            if neighbor != MPI.PROC_NULL:
                requests.append(c.Isend(self.send_h[d], dest=neighbor, tag=100+d))
        return requests

    def exchange_finish(self, requests):
        MPI.Request.Waitall(requests)
        self.recv_d.copy_to_device(self.recv_h, stream=self.boundary)
        unpack_halos_kernel[2,128,self.boundary](self.u, self.recv_d, *self.present)

    def step_serialized(self):
        self.exchange_finish(self.exchange_start())
        tiled_stencil_kernel[(16,16),(16,16),self.boundary](
            self.u, self.out, self.k, self.source, self.dt, 1/DX**2, 0)
        self.boundary.synchronize()
        self.u, self.out = self.out, self.u


def step_overlapped(p):
    # Deep-only launch avoids even speculative tile reads of mutable ghosts.
    tiled_stencil_kernel[(16,16),(16,16),p.interior](
        p.u, p.out, p.k, p.source, p.dt, 1/DX**2, 1)
    requests = p.exchange_start()
    p.exchange_finish(requests)
    tiled_stencil_kernel[(16,16),(16,16),p.boundary](
        p.u, p.out, p.k, p.source, p.dt, 1/DX**2, 2)
    p.interior.synchronize()
    p.boundary.synchronize()
    p.u, p.out = p.out, p.u


class VolatileLoad(ir.instructions.LoadInstr):
    def descr(self, buf):
        out = []
        super().descr(out)
        buf.append(''.join(out).replace('load ', 'load volatile ', 1))


class VolatileStore(ir.instructions.StoreInstr):
    def descr(self, buf):
        out = []
        super().descr(out)
        buf.append(''.join(out).replace('store ', 'store volatile ', 1))


@intrinsic
def volatile_increment(typingctx, array, index):
    signature = types.void(array, index)
    def codegen(context, builder, sig, args):
        ary = context.make_array(sig.args[0])(context, builder, args[0])
        ptr = builder.gep(ary.data, [args[1]])
        load = VolatileLoad(builder.block, ptr, name='counter_read')
        load.align = 8
        builder._insert(load)
        value = builder.add(load, ir.Constant(ir.IntType(64), 1))
        store = VolatileStore(builder.block, value, ptr)
        store.align = 8
        builder._insert(store)
        return context.get_dummy_value()
    return signature, codegen


@njit(nogil=True)
def host_shm_accumulate(shm_view, index, iterations):
    for _ in range(iterations):
        volatile_increment(shm_view, index)


def benchmark_shm_strides(ctx, iterations=SHM_ITERS, repeats=3):
    node, win, view = ctx['node_comm'], ctx['win'], ctx['shm_view']
    rank = node.rank
    host_shm_accumulate(view, rank, 1)
    llvm = host_shm_accumulate.inspect_llvm(host_shm_accumulate.signatures[0])
    audit = {'volatile_load_i64': len(re.findall(r'load volatile i64', llvm)),
             'volatile_store_i64': len(re.findall(r'store volatile i64', llvm)),
             'sha256': hashlib.sha256(llvm.encode()).hexdigest()}
    if not audit['volatile_load_i64'] or not audit['volatile_store_i64']:
        raise AssertionError('Volatile RMW absent from generated LLVM.')
    if ctx['rank'] == 0:
        Path('host_shm_accumulate.ll').write_text(llvm)
    node.Barrier()
    result = {}
    for trial in range(repeats):
        for stride in ((1,8,16), (16,1,8), (8,16,1))[trial % 3]:
            if rank == 0:
                view[:] = 0
            win.Sync()
            node.Barrier()
            win.Sync()
            start = MPI.Wtime()
            host_shm_accumulate(view, rank * stride, iterations)
            elapsed = MPI.Wtime() - start
            win.Sync()
            node.Barrier()
            win.Sync()
            maximum = node.allreduce(elapsed, op=MPI.MAX)
            passed = all(int(view[r*stride]) == iterations for r in range(4))
            if not node.allreduce(passed, op=MPI.LAND):
                raise AssertionError('Shared counter verification failed.')
            result.setdefault(str(stride), []).append(maximum * 1000)
    sweep = {key: {'spacing_bytes': int(key)*8, 'samples_ms': values,
                    'median_ms': float(np.median(values))} for key, values in result.items()}
    sweep['false_sharing_ratio_1_over_8'] = sweep['1']['median_ms']/sweep['8']['median_ms']
    sweep['padding_ratio_8_over_16'] = sweep['8']['median_ms']/sweep['16']['median_ms']
    return sweep, audit


def diagnostics(ctx, p, host_k, host_s, step):
    current = p.u.copy_to_host()
    interior = current[1:257,1:257]
    mass = float(interior.sum() * DX**2)
    energy = float(0.5 * np.square(interior).sum() * DX**2)
    values = np.array([mass, energy], dtype=np.float64)
    total = np.empty_like(values)
    ctx['cart_comm'].Allreduce(values, total, op=MPI.SUM)
    entry = {'step': step, 'mass': float(total[0]), 'energy_l2': float(total[1])}
    if step:
        previous = p.out.copy_to_host()
        old_mass = float(previous[1:257,1:257].sum() * DX**2)
        boundary_flux = 0.0
        # Internal interface flux cancels exactly between neighbor ranks.
        for d, neighbor in enumerate(ctx['neighbors']):
            if neighbor != MPI.PROC_NULL:
                continue
            if d == 0:
                a,b,ka,kb = previous[1,1:257],previous[0,1:257],host_k[1,1:257],host_k[0,1:257]
            elif d == 1:
                a,b,ka,kb = previous[256,1:257],previous[257,1:257],host_k[256,1:257],host_k[257,1:257]
            elif d == 2:
                a,b,ka,kb = previous[1:257,1],previous[1:257,0],host_k[1:257,1],host_k[1:257,0]
            else:
                a,b,ka,kb = previous[1:257,256],previous[1:257,257],host_k[1:257,256],host_k[1:257,257]
            boundary_flux += float((0.5*(ka+kb)*(b-a)).sum())
        expected = p.dt * (boundary_flux + float(host_s[1:257,1:257].sum()*DX**2)
                           - LAMBDA_DAMP*old_mass)
        residual = ctx['cart_comm'].allreduce(mass-old_mass-expected, op=MPI.SUM)
        entry['global_heat_balance_abs_residual'] = abs(residual)
        if abs(residual) >= 1e-12:
            raise AssertionError(f'Distributed heat balance failed: {residual}')
    if ctx['rank'] == 0:
        print('ENERGY', json.dumps(entry), flush=True)
    return entry


def run_mode(ctx, fields, dt, mode):
    p = Pipeline(ctx, *fields, dt)
    action = p.step_serialized if mode == 'serialized' else lambda: step_overlapped(p)
    action()
    p.u.copy_to_device(fields[0])
    p.out.copy_to_device(fields[0])
    cuda.synchronize()
    records = [diagnostics(ctx, p, fields[1], fields[2], 0)]
    times = []
    for endpoint in (100,200,300):
        ctx['cart_comm'].Barrier()
        start = MPI.Wtime()
        for _ in range(100):
            action()
        cuda.synchronize()
        times.append(ctx['cart_comm'].allreduce(MPI.Wtime()-start, op=MPI.MAX))
        records.append(diagnostics(ctx, p, fields[1], fields[2], endpoint))
    return p.u.copy_to_host()[1:257,1:257].copy(), sum(times)*1000, records


def ptx_audit(ctx):
    ptx = tiled_stencil_kernel.inspect_asm(tiled_stencil_kernel.signatures[0])
    counts = {op: len(re.findall(r'\b'+re.escape(op)+r'\b', ptx))
              for op in ('st.shared.f64','ld.shared.f64','bar.sync')}
    # Some LLVM backends express identical 64-bit shared loads/stores as b64/u64.
    counts['shared_stores_any_64bit'] = len(re.findall(r'\bst\.shared\.(?:f64|b64|u64)\b',ptx))
    counts['shared_loads_any_64bit'] = len(re.findall(r'\bld\.shared\.(?:f64|b64|u64)\b',ptx))
    if not counts['bar.sync'] or not counts['shared_stores_any_64bit'] or not counts['shared_loads_any_64bit']:
        raise AssertionError(f'Shared-memory PTX audit failed: {counts}')
    if ctx['rank'] == 0:
        Path('tiled_stencil_kernel.ptx').write_text(ptx)
    counts['sha256'] = hashlib.sha256(ptx.encode()).hexdigest()
    return counts


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('github_username')
    args = parser.parse_args()
    if not re.fullmatch(r'[A-Za-z0-9](?:[A-Za-z0-9-]{0,37}[A-Za-z0-9])?',args.github_username):
        raise ValueError('Set a valid real GitHub username.')
    ctx = init_hybrid_context(args.github_username)
    try:
        fields = initialize_fields(ctx)
        kmax = ctx['cart_comm'].allreduce(float(fields[1].max()),op=MPI.MAX)
        dtmax = DX**2/(4*kmax)
        # Includes damping in the positivity bound, and a 10% safety margin.
        dt = 0.9/(4*kmax/DX**2 + LAMBDA_DAMP)
        serial, serial_ms, serial_energy = run_mode(ctx, fields, dt, 'serialized')
        overlap, overlap_ms, overlap_energy = run_mode(ctx, fields, dt, 'overlapped')
        diff = ctx['cart_comm'].allreduce(float(np.max(np.abs(serial-overlap))), op=MPI.MAX)
        if diff >= 1e-12:
            raise AssertionError(f'Serialized/overlapped mismatch: {diff}')
        audit = ptx_audit(ctx)
        sweep, llvm = benchmark_shm_strides(ctx)
        metadata = {'rank': ctx['rank'], 'coords': ctx['coords'],
                    'cpu_affinity': sorted(os.sched_getaffinity(0)),
                    'shared_base_mod_64': int(ctx['shm_view'].ctypes.data % 64)}
        ranks = ctx['cart_comm'].gather(metadata, root=0)
        if ctx['rank'] == 0:
            import numba, mpi4py
            manifest = dict(schema_version=1, status='executed_on_real_gpu',
                github_username=ctx['github_username'], INSTANCE_SEED=ctx['INSTANCE_SEED'],
                cpu=ctx['cpu'], gpu=ctx['gpu'], ranks=ranks,
                python=sys.version, numpy=np.__version__, numba=numba.__version__,
                mpi4py=mpi4py.__version__, mpi_library=MPI.Get_library_version(),
                global_shape=[512,512], local_shape=[258,258], steps=300,
                boundary_condition='homogeneous Dirichlet', diffusivity='conservative face average',
                source='0.01*(1+0.2*cos(2*pi*x+phase)*cos(2*pi*y))',
                dx=DX, kappa_max=kmax, dt=dt, dt_max=dtmax, courant_ratio=dt/dtmax,
                lambda_damp=LAMBDA_DAMP, max_diff=diff,
                gpu_serialized_ms=serial_ms, gpu_overlapped_ms=overlap_ms,
                overlap_speedup=serial_ms/overlap_ms,
                timing_scope='max-rank synchronized wall time; warmup and diagnostics excluded',
                serialized_energy=serial_energy, overlapped_energy=overlap_energy,
                ptx_audit=audit, llvm_audit=llvm, shm_iterations=SHM_ITERS,
                shm_repeats=3, shm_benchmark=sweep,
                source_sha256=hashlib.sha256(Path(__file__).read_bytes()).hexdigest())
            Path('hybrid_lab_manifest.json').write_text(json.dumps(manifest,indent=2)+'\n')
            print(json.dumps(manifest,indent=2),flush=True)
    finally:
        ctx['win'].Unlock_all()
        ctx['win'].Free()
        ctx['node_comm'].Free()
        ctx['cart_comm'].Free()


if __name__ == '__main__':
    try:
        main()
    except Exception as exc:
        print(f'Rank {MPI.COMM_WORLD.rank}: {type(exc).__name__}: {exc}',file=sys.stderr,flush=True)
        MPI.COMM_WORLD.Abort(1)

Writing hybrid_mpi_cuda_cache.py


In [4]:
# LAB_CELL_4
from pathlib import Path
import subprocess, sys, os, json
Path("hybrid_lab_manifest.json").unlink(missing_ok=True)
cmd = ["mpirun", "--allow-run-as-root", "--oversubscribe", "--bind-to", "none",
       "-np", "4", sys.executable, "-u", "hybrid_mpi_cuda_cache.py", GITHUB_USERNAME]
proc = subprocess.run(cmd, capture_output=True, text=True, env=os.environ.copy())
RUN_LOGS[4] = proc.stdout + ("\nSTDERR:\n" + proc.stderr if proc.stderr else "")
print(RUN_LOGS[4])
if proc.returncode:
    raise RuntimeError(f"MPI/CUDA execution failed: {proc.returncode}")
manifest = json.loads(Path("hybrid_lab_manifest.json").read_text())
assert manifest["status"] == "executed_on_real_gpu"
assert manifest["max_diff"] < 1e-12
assert manifest["steps"] == 300
assert manifest["dt"] <= manifest["dt_max"]

ENERGY {"step": 0, "mass": 0.4074342718652349, "energy_l2": 0.12806495589928038}
ENERGY {"step": 100, "mass": 0.4068610730954346, "energy_l2": 0.12769544584671702, "global_heat_balance_abs_residual": 1.4781571962534795e-17}
ENERGY {"step": 200, "mass": 0.40628852142699234, "energy_l2": 0.12732705920905027, "global_heat_balance_abs_residual": 3.48310535822809e-17}
ENERGY {"step": 300, "mass": 0.40571666488169633, "energy_l2": 0.12695979030196664, "global_heat_balance_abs_residual": 1.2069160707189712e-17}
ENERGY {"step": 0, "mass": 0.4074342718652349, "energy_l2": 0.12806495589928038}
ENERGY {"step": 100, "mass": 0.4068610730954346, "energy_l2": 0.12769544584671702, "global_heat_balance_abs_residual": 1.4781571962534795e-17}
ENERGY {"step": 200, "mass": 0.40628852142699234, "energy_l2": 0.12732705920905027, "global_heat_balance_abs_residual": 3.48310535822809e-17}
ENERGY {"step": 300, "mass": 0.40571666488169633, "energy_l2": 0.12695979030196664, "global_heat_balance_abs_residual": 1.20

In [5]:
# LAB_CELL_5
import json, math
from pathlib import Path
from IPython.display import display, Markdown
m = json.loads(Path("hybrid_lab_manifest.json").read_text())
assert m["status"] == "executed_on_real_gpu", "Real GPU execution is required."
p, h, b = m["ptx_audit"], m["llvm_audit"], m["shm_benchmark"]
energies = "\n".join(f'| {v["step"]} | {v["energy_l2"]:.17g} | {v["mass"]:.17g} | {v.get("global_heat_balance_abs_residual", 0):.5e} |'
                     for v in m["overlapped_energy"])
ANSWER_CELLS = [
f"""### Q1. GPU coalescing versus CPU cache-line granularity
Each float64 is 8 bytes. CPU cache lines hold 8 words (64 bytes); a GPU L2 sector holds 4 words (32 bytes), and a 128-byte line contains 4 sectors. Rank counters with stride 1 occupy one aligned CPU line; strides 8 and 16 separate them by 64 and 128 bytes.

The padded row stride is **258 × 8 = 2,064 bytes**. A 256-element boundary column contains **2,048 useful bytes**, but touches **256 distinct 32-byte GPU sectors = 8,192 bytes**, so an isolated cold gather has **25% sector utilization** (6,144 sector bytes unused). A contiguous, aligned packed column occupies **64 sectors = 2,048 bytes**: 4× fewer sectors for subsequent contiguous access. Both left/right columns have the same sector count; top/bottom rows are contiguous already (unaligned starts can add one sector).

Packing still incurs the strided device reads; it does **not** remove their first-pass sector cost. It makes the D2H/H2D and MPI payload contiguous. If the alternative copied the entire 256 × 258 row-spanning region, it would stage **528,384 bytes** instead of 2,048 bytes per column (258×, 526,336 bytes avoided); a byte-tight span from the first to last column element is 526,328 bytes. These are explicit hypothetical baselines, not measured PCIe counters. Compared with a correct strided gather/copy, PCIe payload reduction need not be 4×. This run stages 8,192 bytes per direction across four strips, with MPI sending only the two existing neighbor strips (4,096 bytes per rank per step). No DRAM/PCIe profiler was used.

Run binding: `{m['github_username']}`, GPU UUID `{m['gpu']['uuid']}`, seed `{m['INSTANCE_SEED']}`.
""",
f"""### Q2. Shared tile synchronization and PTX audit
Exact static counts from this run's `tiled_stencil_kernel.inspect_asm()`:

| PTX instruction | Count |
| --- | ---: |
| `st.shared.f64` | {p['st.shared.f64']} |
| `ld.shared.f64` | {p['ld.shared.f64']} |
| `bar.sync` | {p['bar.sync']} |
| All 64-bit shared stores (`f64/b64/u64`) | {p['shared_stores_any_64bit']} |
| All 64-bit shared loads (`f64/b64/u64`) | {p['shared_loads_any_64bit']} |

The 16 × 16 block has 256 threads, or 8 warps. Threads cooperatively write 324 tile entries, including the halo of the 18 × 18 shared tile. Without `cuda.syncthreads()`, a consumer warp could load a neighboring shared element before its producer warp has stored it: a cross-warp Read-After-Write hazard. All threads reach the barrier before any output eligibility branch. Interior-only tile loads exclude mutable ghost cells, preventing a separate cross-stream read/write race during halo unpack.

Counts are static instruction sites, not executed transaction counts. PTX SHA-256: `{p['sha256']}`. Compiler backends can encode float64 transfers using `b64/u64`; exact `f64` counts are reported even when zero rather than falsified.
""",
f"""### Q3. Dual-stream compute/communication overlap
Measured maximum-rank wall time for 300 synchronized steps, excluding warmup and diagnostics:

| Mode | Time (ms) |
| --- | ---: |
| Serialized | {m['gpu_serialized_ms']:.9f} |
| Overlapped | {m['gpu_overlapped_ms']:.9f} |
| Serialized / overlapped | {m['overlap_speedup']:.6f}× |
| Maximum final-field difference | {m['max_diff']:.17g} |

Deep interior [2..255, 2..255] uses `stream_interior`; packing, pinned staging, MPI exchange, unpacking, and border updates use `stream_boundary`. Receives are posted before sends and tags identify the opposite face. Requests finish before pinned buffers are reused. Each step waits for both streams before swapping arrays.

Four MPI processes select one physical GPU, but normally create **four separate CUDA contexts**, not one shared context. Without MPS, cross-context scheduling and finite copy engines can serialize/limit interprocess overlap; the two streams only express concurrency within each context. All four ranks also compete for the same PCIe link, SMs and DRAM. A small 256² tile workload can make launch, synchronization and MPI overhead dominate, so overlap is not assumed faster. Four dedicated GPUs provide independent compute/copy resources; GPUDirect RDMA can remove CPU staging only with supported hardware, transport and CUDA-aware MPI. This implementation deliberately stages through pinned host memory.
""",
f"""### Q4. MPI-3 shared-memory false sharing
Each rank performs **{m['shm_iterations']:,}** native in-place RMW updates per sample; **{m['shm_repeats']}** samples per stride, rotated order. Values below are medians of the slowest-rank elapsed time in each synchronized trial.

| Stride | Spacing | Median (ms) | Samples (ms) |
| --- | ---: | ---: | --- |
| 1 | 8 B | {b['1']['median_ms']:.9f} | {b['1']['samples_ms']} |
| 8 | 64 B | {b['8']['median_ms']:.9f} | {b['8']['samples_ms']} |
| 16 | 128 B | {b['16']['median_ms']:.9f} | {b['16']['samples_ms']} |

False-sharing ratio **T(stride 1) / T(stride 8) = {b['false_sharing_ratio_1_over_8']:.6f}×**. LLVM audit found **{h['volatile_load_i64']} volatile i64 loads** and **{h['volatile_store_i64']} volatile i64 stores**, with counter totals verified after each trial. Volatile accesses remain in the loop and cannot be hoisted into a single register accumulation. These are ordinary volatile load/store pairs, not atomic operations; each rank exclusively owns its counter word.

An illustrative MESI sequence on distinct coherent cores: both copies start I; Rank 0's RFO gives Rank 0 M and leaves Rank 1 I. Rank 1's RFO invalidates Rank 0 (M→I) and grants Rank 1 M (I→M); Rank 0's next RFO reverses ownership again. If both first read, a clean exclusive copy can start E, downgrade E→S when shared, and then require S→M ownership with the other copy S→I for a write. A read of another core's dirty line can cause M→S under MESI with writeback/intervention. The steady write ping-pong primarily alternates M/I; E/S are not mandatory on every store. Actual x86 protocols may include extra states. No hardware RFO/state trace was collected; this explains the mechanism rather than claiming observed state transitions. Affinity/cpu availability is recorded in the manifest; oversubscribed or SMT-only Colab CPUs can confound a four-core interpretation.
""",
f"""### Q5. 64-byte versus 128-byte padding
Measured **T(stride 8) / T(stride 16) = {b['padding_ratio_8_over_16']:.6f}×**; medians are **{b['8']['median_ms']:.9f} ms** and **{b['16']['median_ms']:.9f} ms**. A ratio >1 favors 128-byte padding; a ratio <1 favors 64-byte padding. Repeat dispersion above matters, and a single ratio does not establish prefetch causality.

Some CPU adjacent-line prefetchers fetch the partner of an even/odd pair of 64-byte lines within an aligned 128-byte region. Stride 8 separates writable words into distinct coherence lines but may still interact through such prefetching. Stride 16 separates them into distinct 128-byte pairs **relative to a 128-byte-aligned origin**; our mandated buffer is only 64-byte aligned, but word spacing of 128 bytes still ensures no pair is shared. This pairing is a prefetch policy, not a universal 128-byte coherence unit. Behavior varies by microarchitecture and configuration; cache-line padding alone is sufficient to remove direct same-line false sharing.

With four counters, stride 1 spans 32 bytes, stride 8 spans 200 bytes, and stride 16 spans 392 bytes; the allocation has 512 usable bytes plus alignment slack. This small sweep does not demonstrate cache-capacity misses. Padding many objects increases footprint, cache/TLB pressure and can induce capacity misses at scale.
""",
f"""### Q6. CFL stability and distributed energy dissipation
The conservative scheme is `u_new = u + dt*(div(kappa*grad(u)) + S - 0.15*u)`, using arithmetic mean diffusivity on shared cell faces and zero physical Dirichlet ghosts.

From this run, **κmax = {m['kappa_max']:.17g}**, **Δx = 1/513 = {m['dx']:.17g}**.

`Δtmax = Δx²/(4 κmax) = {m['dt_max']:.17g}`.

Chosen **Δt = {m['dt']:.17g}** and **Δt/Δtmax = {m['courant_ratio']:.12f} < 1**. The implementation additionally includes damping in its positivity bound, `dt = 0.9/(4*kappa_max/dx² + 0.15)`.

Printed `Allreduce` diagnostics for the overlapped run:

| Step | E = ½ Δx² Σu² | Heat M = Δx² Σu | Absolute global heat-balance residual |
| --- | ---: | ---: | ---: |
{energies}

Only rank-local arrays are copied for diagnostics; no global field gather/Allgather occurs. Internal interface fluxes cancel under conservative face averaging. With nonzero source, damping and open Dirichlet boundaries, heat is **not constant**: the checked conservation statement is `M_new - M_old = dt*(boundary flux + total source - 0.15*M_old)`. Residuals above test this physical balance on the checkpoint step. Energy E measures dissipation, while source forcing can add energy; energy conservation is not asserted for this dissipative PDE. Serialized diagnostics are also retained in the manifest.
"""
]
for answer in ANSWER_CELLS:
    display(Markdown(answer))

### Q1. GPU coalescing versus CPU cache-line granularity
Each float64 is 8 bytes. CPU cache lines hold 8 words (64 bytes); a GPU L2 sector holds 4 words (32 bytes), and a 128-byte line contains 4 sectors. Rank counters with stride 1 occupy one aligned CPU line; strides 8 and 16 separate them by 64 and 128 bytes.

The padded row stride is **258 × 8 = 2,064 bytes**. A 256-element boundary column contains **2,048 useful bytes**, but touches **256 distinct 32-byte GPU sectors = 8,192 bytes**, so an isolated cold gather has **25% sector utilization** (6,144 sector bytes unused). A contiguous, aligned packed column occupies **64 sectors = 2,048 bytes**: 4× fewer sectors for subsequent contiguous access. Both left/right columns have the same sector count; top/bottom rows are contiguous already (unaligned starts can add one sector).

Packing still incurs the strided device reads; it does **not** remove their first-pass sector cost. It makes the D2H/H2D and MPI payload contiguous. If the alternative copied the entire 256 × 258 row-spanning region, it would stage **528,384 bytes** instead of 2,048 bytes per column (258×, 526,336 bytes avoided); a byte-tight span from the first to last column element is 526,328 bytes. These are explicit hypothetical baselines, not measured PCIe counters. Compared with a correct strided gather/copy, PCIe payload reduction need not be 4×. This run stages 8,192 bytes per direction across four strips, with MPI sending only the two existing neighbor strips (4,096 bytes per rank per step). No DRAM/PCIe profiler was used.

Run binding: `240103149-lgtm`, GPU UUID `GPU-639848cb-ba85-af70-c282-e9deed9ac09b`, seed `810338268274559305`.


### Q2. Shared tile synchronization and PTX audit
Exact static counts from this run's `tiled_stencil_kernel.inspect_asm()`:

| PTX instruction | Count |
| --- | ---: |
| `st.shared.f64` | 8 |
| `ld.shared.f64` | 5 |
| `bar.sync` | 1 |
| All 64-bit shared stores (`f64/b64/u64`) | 11 |
| All 64-bit shared loads (`f64/b64/u64`) | 5 |

The 16 × 16 block has 256 threads, or 8 warps. Threads cooperatively write 324 tile entries, including the halo of the 18 × 18 shared tile. Without `cuda.syncthreads()`, a consumer warp could load a neighboring shared element before its producer warp has stored it: a cross-warp Read-After-Write hazard. All threads reach the barrier before any output eligibility branch. Interior-only tile loads exclude mutable ghost cells, preventing a separate cross-stream read/write race during halo unpack.

Counts are static instruction sites, not executed transaction counts. PTX SHA-256: `49928782c63b8e191a58f5782c79f83e34625b5fd4b19f53e59c881ae02e6228`. Compiler backends can encode float64 transfers using `b64/u64`; exact `f64` counts are reported even when zero rather than falsified.


### Q3. Dual-stream compute/communication overlap
Measured maximum-rank wall time for 300 synchronized steps, excluding warmup and diagnostics:

| Mode | Time (ms) |
| --- | ---: |
| Serialized | 1675.458197000 |
| Overlapped | 2042.741241000 |
| Serialized / overlapped | 0.820201× |
| Maximum final-field difference | 0 |

Deep interior [2..255, 2..255] uses `stream_interior`; packing, pinned staging, MPI exchange, unpacking, and border updates use `stream_boundary`. Receives are posted before sends and tags identify the opposite face. Requests finish before pinned buffers are reused. Each step waits for both streams before swapping arrays.

Four MPI processes select one physical GPU, but normally create **four separate CUDA contexts**, not one shared context. Without MPS, cross-context scheduling and finite copy engines can serialize/limit interprocess overlap; the two streams only express concurrency within each context. All four ranks also compete for the same PCIe link, SMs and DRAM. A small 256² tile workload can make launch, synchronization and MPI overhead dominate, so overlap is not assumed faster. Four dedicated GPUs provide independent compute/copy resources; GPUDirect RDMA can remove CPU staging only with supported hardware, transport and CUDA-aware MPI. This implementation deliberately stages through pinned host memory.


### Q4. MPI-3 shared-memory false sharing
Each rank performs **40,000,000** native in-place RMW updates per sample; **3** samples per stride, rotated order. Values below are medians of the slowest-rank elapsed time in each synchronized trial.

| Stride | Spacing | Median (ms) | Samples (ms) |
| --- | ---: | ---: | --- |
| 1 | 8 B | 408.126398000 | [408.1263980000003, 394.7064899999999, 423.35113000000035] |
| 8 | 64 B | 272.471076000 | [300.38860600000027, 244.1533869999999, 272.47107600000044] |
| 16 | 128 B | 254.070622000 | [231.74559100000013, 254.07062199999953, 301.5936499999992] |

False-sharing ratio **T(stride 1) / T(stride 8) = 1.497871×**. LLVM audit found **9 volatile i64 loads** and **9 volatile i64 stores**, with counter totals verified after each trial. Volatile accesses remain in the loop and cannot be hoisted into a single register accumulation. These are ordinary volatile load/store pairs, not atomic operations; each rank exclusively owns its counter word.

An illustrative MESI sequence on distinct coherent cores: both copies start I; Rank 0's RFO gives Rank 0 M and leaves Rank 1 I. Rank 1's RFO invalidates Rank 0 (M→I) and grants Rank 1 M (I→M); Rank 0's next RFO reverses ownership again. If both first read, a clean exclusive copy can start E, downgrade E→S when shared, and then require S→M ownership with the other copy S→I for a write. A read of another core's dirty line can cause M→S under MESI with writeback/intervention. The steady write ping-pong primarily alternates M/I; E/S are not mandatory on every store. Actual x86 protocols may include extra states. No hardware RFO/state trace was collected; this explains the mechanism rather than claiming observed state transitions. Affinity/cpu availability is recorded in the manifest; oversubscribed or SMT-only Colab CPUs can confound a four-core interpretation.


### Q5. 64-byte versus 128-byte padding
Measured **T(stride 8) / T(stride 16) = 1.072423×**; medians are **272.471076000 ms** and **254.070622000 ms**. A ratio >1 favors 128-byte padding; a ratio <1 favors 64-byte padding. Repeat dispersion above matters, and a single ratio does not establish prefetch causality.

Some CPU adjacent-line prefetchers fetch the partner of an even/odd pair of 64-byte lines within an aligned 128-byte region. Stride 8 separates writable words into distinct coherence lines but may still interact through such prefetching. Stride 16 separates them into distinct 128-byte pairs **relative to a 128-byte-aligned origin**; our mandated buffer is only 64-byte aligned, but word spacing of 128 bytes still ensures no pair is shared. This pairing is a prefetch policy, not a universal 128-byte coherence unit. Behavior varies by microarchitecture and configuration; cache-line padding alone is sufficient to remove direct same-line false sharing.

With four counters, stride 1 spans 32 bytes, stride 8 spans 200 bytes, and stride 16 spans 392 bytes; the allocation has 512 usable bytes plus alignment slack. This small sweep does not demonstrate cache-capacity misses. Padding many objects increases footprint, cache/TLB pressure and can induce capacity misses at scale.


### Q6. CFL stability and distributed energy dissipation
The conservative scheme is `u_new = u + dt*(div(kappa*grad(u)) + S - 0.15*u)`, using arithmetic mean diffusivity on shared cell faces and zero physical Dirichlet ghosts.

From this run, **κmax = 0.12499985603552494**, **Δx = 1/513 = 0.0019493177387914229**.

`Δtmax = Δx²/(4 κmax) = 7.5996880462146131e-06`.

Chosen **Δt = 6.8397114446421564e-06** and **Δt/Δtmax = 0.899998974043 < 1**. The implementation additionally includes damping in its positivity bound, `dt = 0.9/(4*kappa_max/dx² + 0.15)`.

Printed `Allreduce` diagnostics for the overlapped run:

| Step | E = ½ Δx² Σu² | Heat M = Δx² Σu | Absolute global heat-balance residual |
| --- | ---: | ---: | ---: |
| 0 | 0.12806495589928038 | 0.4074342718652349 | 0.00000e+00 |
| 100 | 0.12769544584671702 | 0.40686107309543462 | 1.47816e-17 |
| 200 | 0.12732705920905027 | 0.40628852142699234 | 3.48311e-17 |
| 300 | 0.12695979030196664 | 0.40571666488169633 | 1.20692e-17 |

Only rank-local arrays are copied for diagnostics; no global field gather/Allgather occurs. Internal interface fluxes cancel under conservative face averaging. With nonzero source, damping and open Dirichlet boundaries, heat is **not constant**: the checked conservation statement is `M_new - M_old = dt*(boundary flux + total source - 0.15*M_old)`. Residuals above test this physical balance on the checkpoint step. Energy E measures dissipation, while source forcing can add energy; energy conservation is not asserted for this dissipative PDE. Serialized diagnostics are also retained in the manifest.


In [6]:
# LAB_CELL_6
import nbformat as nbf, zipfile, hashlib
from pathlib import Path
history = get_ipython().history_manager.input_hist_raw
selected = {}
for source in history:
    for n in (1, 2, 4, 5, 6):
        if source.startswith(f"# LAB_CELL_{n}\n"):
            selected[n] = source
    if source.startswith("%%writefile hybrid_mpi_cuda_cache.py"):
        selected[3] = source
assert all(n in selected for n in (1, 2, 3, 4, 5, 6)), "Run all notebook cells in order."
nb = nbf.v4.new_notebook()
nb.metadata.update({"kernelspec": {"display_name":"Python 3", "language":"python", "name":"python3"},
                    "accelerator":"GPU"})
nb.cells.append(nbf.v4.new_markdown_cell("# Hybrid MPI–CUDA & Cache-Line Coherency\nExecuted hardware evidence and analytical responses. GitHub: " + GITHUB_USERNAME))
for n in (1, 2, 3, 4, 5, 6):
    cell = nbf.v4.new_code_cell(selected[n], execution_count=n)
    if n in RUN_LOGS:
        cell.outputs = [nbf.v4.new_output("stream", name="stdout", text=RUN_LOGS[n])]
    elif n == 3:
        cell.outputs = [nbf.v4.new_output("stream", name="stdout", text="Writing hybrid_mpi_cuda_cache.py\n")]
    elif n == 5:
        cell.outputs = [nbf.v4.new_output("display_data", data={"text/markdown": answer}, metadata={}) for answer in ANSWER_CELLS]
    elif n == 6:
        cell.outputs = [nbf.v4.new_output("stream", name="stdout", text="Exported notebook, real manifest, source and assembly audits.\n")]
    nb.cells.append(cell)
for answer in ANSWER_CELLS:
    nb.cells.append(nbf.v4.new_markdown_cell(answer))
nbf.validate(nb)
nbf.write(nb, "hybrid_mpi_cuda_cache_lab.ipynb")
assert hashlib.sha256(Path("hybrid_mpi_cuda_cache.py").read_bytes()).hexdigest() == m["source_sha256"]
with zipfile.ZipFile("hybrid_mpi_cuda_cache_submission.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for name in ("hybrid_mpi_cuda_cache_lab.ipynb", "hybrid_mpi_cuda_cache.py", "hybrid_lab_manifest.json",
                 "host_shm_accumulate.ll", "tiled_stencil_kernel.ptx"):
        z.write(name, arcname=name)
print("Exported notebook, real manifest, source and assembly audits.")
from google.colab import files
files.download("hybrid_mpi_cuda_cache_submission.zip")

Exported notebook, real manifest, source and assembly audits.


### Q1. GPU coalescing versus CPU cache-line granularity
Each float64 is 8 bytes. CPU cache lines hold 8 words (64 bytes); a GPU L2 sector holds 4 words (32 bytes), and a 128-byte line contains 4 sectors. Rank counters with stride 1 occupy one aligned CPU line; strides 8 and 16 separate them by 64 and 128 bytes.

The padded row stride is **258 × 8 = 2,064 bytes**. A 256-element boundary column contains **2,048 useful bytes**, but touches **256 distinct 32-byte GPU sectors = 8,192 bytes**, so an isolated cold gather has **25% sector utilization** (6,144 sector bytes unused). A contiguous, aligned packed column occupies **64 sectors = 2,048 bytes**: 4× fewer sectors for subsequent contiguous access. Both left/right columns have the same sector count; top/bottom rows are contiguous already (unaligned starts can add one sector).

Packing still incurs the strided device reads; it does **not** remove their first-pass sector cost. It makes the D2H/H2D and MPI payload contiguous. If the alternative copied the entire 256 × 258 row-spanning region, it would stage **528,384 bytes** instead of 2,048 bytes per column (258×, 526,336 bytes avoided); a byte-tight span from the first to last column element is 526,328 bytes. These are explicit hypothetical baselines, not measured PCIe counters. Compared with a correct strided gather/copy, PCIe payload reduction need not be 4×. This run stages 8,192 bytes per direction across four strips, with MPI sending only the two existing neighbor strips (4,096 bytes per rank per step). No DRAM/PCIe profiler was used.

Run binding: `240103149-lgtm`, GPU UUID `GPU-639848cb-ba85-af70-c282-e9deed9ac09b`, seed `810338268274559305`.


### Q2. Shared tile synchronization and PTX audit
Exact static counts from this run's `tiled_stencil_kernel.inspect_asm()`:

| PTX instruction | Count |
| --- | ---: |
| `st.shared.f64` | 8 |
| `ld.shared.f64` | 5 |
| `bar.sync` | 1 |
| All 64-bit shared stores (`f64/b64/u64`) | 11 |
| All 64-bit shared loads (`f64/b64/u64`) | 5 |

The 16 × 16 block has 256 threads, or 8 warps. Threads cooperatively write 324 tile entries, including the halo of the 18 × 18 shared tile. Without `cuda.syncthreads()`, a consumer warp could load a neighboring shared element before its producer warp has stored it: a cross-warp Read-After-Write hazard. All threads reach the barrier before any output eligibility branch. Interior-only tile loads exclude mutable ghost cells, preventing a separate cross-stream read/write race during halo unpack.

Counts are static instruction sites, not executed transaction counts. PTX SHA-256: `49928782c63b8e191a58f5782c79f83e34625b5fd4b19f53e59c881ae02e6228`. Compiler backends can encode float64 transfers using `b64/u64`; exact `f64` counts are reported even when zero rather than falsified.


### Q3. Dual-stream compute/communication overlap
Measured maximum-rank wall time for 300 synchronized steps, excluding warmup and diagnostics:

| Mode | Time (ms) |
| --- | ---: |
| Serialized | 1675.458197000 |
| Overlapped | 2042.741241000 |
| Serialized / overlapped | 0.820201× |
| Maximum final-field difference | 0 |

Deep interior [2..255, 2..255] uses `stream_interior`; packing, pinned staging, MPI exchange, unpacking, and border updates use `stream_boundary`. Receives are posted before sends and tags identify the opposite face. Requests finish before pinned buffers are reused. Each step waits for both streams before swapping arrays.

Four MPI processes select one physical GPU, but normally create **four separate CUDA contexts**, not one shared context. Without MPS, cross-context scheduling and finite copy engines can serialize/limit interprocess overlap; the two streams only express concurrency within each context. All four ranks also compete for the same PCIe link, SMs and DRAM. A small 256² tile workload can make launch, synchronization and MPI overhead dominate, so overlap is not assumed faster. Four dedicated GPUs provide independent compute/copy resources; GPUDirect RDMA can remove CPU staging only with supported hardware, transport and CUDA-aware MPI. This implementation deliberately stages through pinned host memory.


### Q4. MPI-3 shared-memory false sharing
Each rank performs **40,000,000** native in-place RMW updates per sample; **3** samples per stride, rotated order. Values below are medians of the slowest-rank elapsed time in each synchronized trial.

| Stride | Spacing | Median (ms) | Samples (ms) |
| --- | ---: | ---: | --- |
| 1 | 8 B | 408.126398000 | [408.1263980000003, 394.7064899999999, 423.35113000000035] |
| 8 | 64 B | 272.471076000 | [300.38860600000027, 244.1533869999999, 272.47107600000044] |
| 16 | 128 B | 254.070622000 | [231.74559100000013, 254.07062199999953, 301.5936499999992] |

False-sharing ratio **T(stride 1) / T(stride 8) = 1.497871×**. LLVM audit found **9 volatile i64 loads** and **9 volatile i64 stores**, with counter totals verified after each trial. Volatile accesses remain in the loop and cannot be hoisted into a single register accumulation. These are ordinary volatile load/store pairs, not atomic operations; each rank exclusively owns its counter word.

An illustrative MESI sequence on distinct coherent cores: both copies start I; Rank 0's RFO gives Rank 0 M and leaves Rank 1 I. Rank 1's RFO invalidates Rank 0 (M→I) and grants Rank 1 M (I→M); Rank 0's next RFO reverses ownership again. If both first read, a clean exclusive copy can start E, downgrade E→S when shared, and then require S→M ownership with the other copy S→I for a write. A read of another core's dirty line can cause M→S under MESI with writeback/intervention. The steady write ping-pong primarily alternates M/I; E/S are not mandatory on every store. Actual x86 protocols may include extra states. No hardware RFO/state trace was collected; this explains the mechanism rather than claiming observed state transitions. Affinity/cpu availability is recorded in the manifest; oversubscribed or SMT-only Colab CPUs can confound a four-core interpretation.


### Q5. 64-byte versus 128-byte padding
Measured **T(stride 8) / T(stride 16) = 1.072423×**; medians are **272.471076000 ms** and **254.070622000 ms**. A ratio >1 favors 128-byte padding; a ratio <1 favors 64-byte padding. Repeat dispersion above matters, and a single ratio does not establish prefetch causality.

Some CPU adjacent-line prefetchers fetch the partner of an even/odd pair of 64-byte lines within an aligned 128-byte region. Stride 8 separates writable words into distinct coherence lines but may still interact through such prefetching. Stride 16 separates them into distinct 128-byte pairs **relative to a 128-byte-aligned origin**; our mandated buffer is only 64-byte aligned, but word spacing of 128 bytes still ensures no pair is shared. This pairing is a prefetch policy, not a universal 128-byte coherence unit. Behavior varies by microarchitecture and configuration; cache-line padding alone is sufficient to remove direct same-line false sharing.

With four counters, stride 1 spans 32 bytes, stride 8 spans 200 bytes, and stride 16 spans 392 bytes; the allocation has 512 usable bytes plus alignment slack. This small sweep does not demonstrate cache-capacity misses. Padding many objects increases footprint, cache/TLB pressure and can induce capacity misses at scale.


### Q6. CFL stability and distributed energy dissipation
The conservative scheme is `u_new = u + dt*(div(kappa*grad(u)) + S - 0.15*u)`, using arithmetic mean diffusivity on shared cell faces and zero physical Dirichlet ghosts.

From this run, **κmax = 0.12499985603552494**, **Δx = 1/513 = 0.0019493177387914229**.

`Δtmax = Δx²/(4 κmax) = 7.5996880462146131e-06`.

Chosen **Δt = 6.8397114446421564e-06** and **Δt/Δtmax = 0.899998974043 < 1**. The implementation additionally includes damping in its positivity bound, `dt = 0.9/(4*kappa_max/dx² + 0.15)`.

Printed `Allreduce` diagnostics for the overlapped run:

| Step | E = ½ Δx² Σu² | Heat M = Δx² Σu | Absolute global heat-balance residual |
| --- | ---: | ---: | ---: |
| 0 | 0.12806495589928038 | 0.4074342718652349 | 0.00000e+00 |
| 100 | 0.12769544584671702 | 0.40686107309543462 | 1.47816e-17 |
| 200 | 0.12732705920905027 | 0.40628852142699234 | 3.48311e-17 |
| 300 | 0.12695979030196664 | 0.40571666488169633 | 1.20692e-17 |

Only rank-local arrays are copied for diagnostics; no global field gather/Allgather occurs. Internal interface fluxes cancel under conservative face averaging. With nonzero source, damping and open Dirichlet boundaries, heat is **not constant**: the checked conservation statement is `M_new - M_old = dt*(boundary flux + total source - 0.15*M_old)`. Residuals above test this physical balance on the checkpoint step. Energy E measures dissipation, while source forcing can add energy; energy conservation is not asserted for this dissipative PDE. Serialized diagnostics are also retained in the manifest.
